## 统计分析

通过指定统计分析字段，得到每个特征的p_value，所有的p_value计算都是基于Ttest计算。支持指定不同的分组`group`，例如train、val、test等分组统计。

对于两大类不同的特征

1. 离散特征，统计数量以及占比。
2. 连续特征，统计均值、方差。

In [ ]:
import pandas as pd
import numpy as np
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.utils import print_join_info

task = get_param_in_cwd('task_column') or 'label'
p_value = get_param_in_cwd('p_value') or 0.05
# 修改成自己临床数据的文件。
test_data = pd.read_csv(get_param_in_cwd('clinic_file'))
stats_columns_settings = get_param_in_cwd('stats_columns')
continuous_columns_settings = get_param_in_cwd('continuous_columns')
mapping_columns_settings = get_param_in_cwd('mapping_columns')
test_data = test_data[[c for c in test_data.columns if c != task]]
test_data['ID'] = test_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
group_info = pd.read_csv(get_param_in_cwd('label_file'))

print_join_info(test_data, group_info)
test_data = pd.merge(test_data, group_info, on='ID', how='inner')
test_data

In [ ]:
test_data['group'].value_counts()

# 特征名称处理

去掉所有特征名称中的特殊字符。

In [ ]:
import re

def map_cnames(x):
    x = re.split('[（|(]', x)[0]
    x = x.replace('-', '_').replace(' ', '_').replace('>', '').replace('/', '_').replace('+', '_')
    return x.strip('_')

test_data.columns = list(map(map_cnames, test_data.columns))
test_data.columns

In [ ]:
def map_state(x):
    if x.startswith('IV'):
        return 4
    elif x.startswith('III'):
        return 3
    elif x.startswith('II'):
        return 2
    elif x.startswith('I'):
        return 1
    return 0
test_data['FIGO_stage'] = test_data['FIGO_stage'].map(lambda x: map_state(x))
test_data

# 分析数据

获取待分析的特征列名，如未制定，自动侦测。

In [ ]:
stats_columns = [c for c in stats_columns_settings or list(test_data.columns[1:-2]) if c not in ['TRG', 'ypT_stage', 'ypTNM_stage']]
test_data = test_data.copy()[['ID'] + stats_columns + ['group', task]]
test_data#['group'].value_counts()

# 特征队列映射

所有需要进行特征映射的队列，range未制定，可以进行自动判断。

In [ ]:
mapping_columns = mapping_columns_settings or [c for c in test_data.columns[1:-2] if test_data[c].dtype == object]
mapping_columns

# 数据映射

针对所有非数值形式的数据，可以进行类别映射。

In [ ]:
from onekey_algo.custom.utils import map2numerical

data, mapping = map2numerical(test_data, mapping_columns=mapping_columns)
mapping

In [ ]:
data.dtypes

# 连续特征列

自动识别所有可能的连续特征列。如果列不是整数，或者列的元素超过5个，则呗认定为连续特征。

In [ ]:
from onekey_algo.custom.components.comp1 import fillna

test_data = fillna(test_data, fill_mod='50%')
continuous_columns = []
for col in test_data.columns:
    if test_data[col].apply(lambda x: x.is_integer() if isinstance(x, float) else False).all():
        test_data[col] = test_data[col].astype(int)

for c in stats_columns:
#     print(c, np.unique(test_data[c]), test_data[c].dtype)
    if len(np.unique(test_data[c])) > 8 or not np.int8 <= test_data[c].dtype <= np.int64:
        continuous_columns.append(c)
        
continuous_columns = continuous_columns_settings or continuous_columns
continuous_columns = [c for c in continuous_columns if c not in ('')]
continuous_columns

# 缺失值填充

In [ ]:
import os
os.makedirs('data', exist_ok=True)
data = test_data
data.to_csv('data/clinical.csv', index=False)
data

### 统计分析

支持两种格式数据，分别对应`pretty`参数的`True`和`False`, 当为`True`时，输出的是表格模式，反之则为dict数据。

```python
def clinic_stats(data: DataFrame, stats_columns: Union[str, List[str]], label_column='label',
                 group_column: str = None, continuous_columns: Union[str, List[str]] = None,
                 pretty: bool = True) -> Union[dict, DataFrame]:
    """

    Args:
        data: 数据
        stats_columns: 需要统计的列名
        label_column: 二分类的标签列，默认`label`
        group_column: 分组统计依据，例如区分训练组、测试组、验证组。
        continuous_columns: 那些列是连续变量，连续变量统计均值方差。
        pretty: bool, 是否对结果进行格式美化。

    Returns:
        stats DataFrame or json

    """
```

In [ ]:
from onekey_algo.custom.components.stats import clinic_stats

pd.set_option('display.max_rows', None)
stats = clinic_stats(data, 
                     stats_columns= stats_columns,
                     label_column=task, 
                     group_column='group', 
                     continuous_columns= continuous_columns, 
                     pretty=True, verbose=False)
stats.to_csv('data/stats.csv', index=False, encoding='utf_8_sig')
stats

In [ ]:
from onekey_algo.custom.components.stats import clinic_stats

pd.set_option('display.max_rows', None)
stats_train_val = clinic_stats(data[data['group'].isin(['train', 'val'])], 
                               stats_columns= stats_columns,
                               label_column='group', 
                               group_column=None, 
                               continuous_columns= continuous_columns, 
                               pretty=True, verbose=False).reset_index(drop=True)

stats = clinic_stats(data, 
                     stats_columns= stats_columns,
                     label_column='group', 
                     group_column=None, 
                     continuous_columns= continuous_columns, 
                     pretty=True, verbose=False)
# display(stats)
for subset in get_param_in_cwd('subsets'):
    if subset not in ['train', 'val']:
        stats_train_val[subset] = stats[f'-label={subset}']
# stats_train_val['test2'] = stats['-label=test2']
stats_train_val.to_csv('data/stats_all.csv', index=False, encoding='utf_8_sig')
stats_train_val

In [ ]:
from onekey_algo.custom.components.comp1 import uni_multi_variable_analysis                        

r = uni_multi_variable_analysis(data[data['group'] == 'train'], stats_columns, save_dir='img', p_value4multi=p_value, 
                                hazard_ratios=True, label_column=task, algo='ols')

In [ ]:
uni_v = pd.read_csv('img/multivariable_reg.csv')
uni_v = uni_v[uni_v['p_value'] <= 0.05]
sel_data = data[['ID'] + list(uni_v['feature_name']) + ['group', task]]
sel_data.to_csv('data/clinical_sel.csv', index=False)
sel_data

In [ ]:
uni = pd.read_csv('img/univariable_reg.csv')
uni = uni[[c for c in uni if ('OR' in c and 'Log' not in c) or c in ['feature_name', 'p_value']]]
uni['95% CI'] = [f"{y:.4f}-{z:.4f}" for x, y, z in np.array(uni[[c for c in uni.columns if c not in ['feature_name', 'p_value']]])]
multi = pd.read_csv('img/multivariable_reg.csv')
multi = multi[[c for c in uni if ('OR' in c and 'Log' not in c) or c in ['feature_name', 'p_value']]]
multi['95% CI'] = [f"{y:.4f}-{z:.4f}" for x, y, z in np.array(multi[[c for c in multi.columns if c not in ['feature_name', 'p_value']]])]
info = pd.merge(uni[['feature_name', 'OR', '95% CI', 'p_value']], multi[['feature_name' , 'OR', '95% CI', 'p_value']],
                on='feature_name', how='left', suffixes=['_UNI', '_MULTI']).applymap(lambda x: '' if pd.isna(x) else x)
info.applymap(lambda x: x if isinstance(x, str) or x > 0.05 else '<0.05')